# Hands-On Lab: Build an Interactive Dashboard with Plotly Dash

In this lab, you will build a Plotly Dash application for users to perform interactive visual analytics on SpaceX launch data in real-time.

This dashboard application contains input components such as a dropdown list and a range slider to interact with a pie chart and a scatter point chart.

### Objectives:
1. **TASK 1:** Add a Launch Site Drop-down Input Component
2. **TASK 2:** Add a callback function to render `success-pie-chart` based on selected site dropdown
3. **TASK 3:** Add a Range Slider to Select Payload
4. **TASK 4:** Add a callback function to render the `success-payload-scatter-chart` scatter plot
5. **TASK 5:** Analyze visual patterns and answer the 5 core business questions

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import urllib.request
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Download and load the dataset
csv_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
csv_file = "spacex_launch_dash.csv"

if not os.path.exists(csv_file):
    urllib.request.urlretrieve(csv_url, csv_file)

spacex_df = pd.read_csv(csv_file)
print("Dataset shape:", spacex_df.shape)
spacex_df.head()

Dataset shape: (56, 7)


,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [3]:
# Dataset statistics and launch site distribution
min_payload = spacex_df['Payload Mass (kg)'].min()
max_payload = spacex_df['Payload Mass (kg)'].max()

print(f"Payload Mass Range: {min_payload} kg to {max_payload} kg")
print("\nLaunch sites available:")
print(spacex_df['Launch Site'].value_counts())

print("\nSuccess vs. Failure breakdown by site:")
print(spacex_df.groupby(['Launch Site', 'class']).size().unstack(fill_value=0))

Payload Mass Range: 0.0 kg to 9600.0 kg

Launch sites available:
Launch Site
CCAFS LC-40     26
KSC LC-39A      13
VAFB SLC-4E     10
CCAFS SLC-40     7
Name: count, dtype: int64

Success vs. Failure breakdown by site:
class          0   1
Launch Site         
CCAFS LC-40   19   7
CCAFS SLC-40   4   3
KSC LC-39A     3  10
VAFB SLC-4E    6   4


## TASK 1: Add a Launch Site Drop-down Input Component

We configure a `dcc.Dropdown` component with:
- `id='site-dropdown'`
- Options including an `'ALL'` sites choice and individual launch sites
- Default `value='ALL'`
- Searchable enabled (`searchable=True`)

In [4]:
# Build dropdown options dynamically from unique launch sites
launch_sites = spacex_df['Launch Site'].unique()
dropdown_options = [{'label': 'All Sites', 'value': 'ALL'}]
for site in launch_sites:
    dropdown_options.append({'label': site, 'value': site})

dropdown_component = dcc.Dropdown(
    id='site-dropdown',
    options=dropdown_options,
    value='ALL',
    placeholder="Select a Launch Site here",
    searchable=True
)

print("Dropdown options:")
for opt in dropdown_options:
    print(opt)

Dropdown options:
{'label': 'All Sites', 'value': 'ALL'}
{'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'}
{'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'}
{'label': 'KSC LC-39A', 'value': 'KSC LC-39A'}
{'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}


## TASK 2: Add a Callback Function to Render Success Pie Chart

The callback function receives the selected site from `site-dropdown` and returns a Plotly pie chart:
- When **ALL** sites are selected: calculates total successful launches grouped by launch site.
- When a **specific** site is selected: filters the dataframe for that site and displays the proportion of Success (`class=1`) vs. Failure (`class=0`).

In [5]:
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df,
            values='class',
            names='Launch Site',
            title='Total Success Launches By Site'
        )
        return fig
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        site_counts = filtered_df['class'].value_counts().reset_index()
        site_counts.columns = ['class', 'count']
        site_counts['class_label'] = site_counts['class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(
            site_counts,
            values='count',
            names='class_label',
            title=f'Total Success Launches for site {entered_site}',
            color='class_label',
            color_discrete_map={'Success': '#2ecc71', 'Failure': '#e74c3c'}
        )
        return fig

# Demonstrate Pie Chart for ALL sites
fig_all = get_pie_chart('ALL')
fig_all.show()

In [6]:
# Demonstrate Pie Chart for specific site: KSC LC-39A
fig_site = get_pie_chart('KSC LC-39A')
fig_site.show()

## TASK 3: Add a Range Slider to Select Payload

We configure a `dcc.RangeSlider` component with:
- `id='payload-slider'`
- `min=0`, `max=10000`, `step=1000`
- Value set to `[min_payload, max_payload]`
- Marks at every 2500 kg interval

In [7]:
payload_slider = dcc.RangeSlider(
    id='payload-slider',
    min=0,
    max=10000,
    step=1000,
    marks={
        0: '0 kg',
        2500: '2500 kg',
        5000: '5000 kg',
        7500: '7500 kg',
        10000: '10000 kg'
    },
    value=[min_payload, max_payload]
)
print("RangeSlider configured with payload boundaries:", [min_payload, max_payload])

RangeSlider configured with payload boundaries: [np.float64(0.0), np.float64(9600.0)]


## TASK 4: Add a Callback Function to Render Success Payload Scatter Chart

The callback function receives both `site-dropdown` and `payload-slider` as inputs:
- Filters `spacex_df` within the selected payload mass boundaries.
- Plots `Payload Mass (kg)` on the X-axis and launch outcome `class` on the Y-axis.
- Colors data points by `Booster Version Category` to observe performance differences across rocket generations.

In [8]:
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) & 
                            (spacex_df['Payload Mass (kg)'] <= high)]
    
    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Correlation between Payload and Success for all Sites',
            hover_data=['Booster Version', 'Launch Site']
        )
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(
            site_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Correlation between Payload and Success for {entered_site}',
            hover_data=['Booster Version']
        )
        return fig

# Demonstrate Scatter Plot for ALL sites
fig_scatter_all = get_scatter_chart('ALL', [min_payload, max_payload])
fig_scatter_all.show()

In [9]:
# Demonstrate Scatter Plot for KSC LC-39A within payload range 2000 - 8000 kg
fig_scatter_ksc = get_scatter_chart('KSC LC-39A', [2000, 8000])
fig_scatter_ksc.show()

## TASK 5: Full Dash Application Definition

Below is the assembled Dash application structure. It can be executed inside this notebook or run as a standalone script via `python spacex_dash_app.py`.

In [10]:
# Instantiate Dash app
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    
    dcc.Dropdown(
        id='site-dropdown',
        options=dropdown_options,
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    html.Br(),

    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P("Payload range (Kg):"),
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={
            0: '0',
            2500: '2500',
            5000: '5000',
            7500: '7500',
            10000: '10000'
        },
        value=[min_payload, max_payload]
    ),
    html.Br(),

    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

# Register callbacks
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def update_pie_chart(site):
    return get_pie_chart(site)

@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def update_scatter_chart(site, payload):
    return get_scatter_chart(site, payload)

print("Dash application layout and callbacks successfully constructed!")

Dash application layout and callbacks successfully constructed!


## Visual Analytics Findings & Insights

Based on the interactive dashboard visualizations, we answer the 5 core business questions:

1. **Which site has the largest successful launches?**
   - **KSC LC-39A** has the largest number of successful launches with **10 successful missions** (representing 41.7% of all successful launches across all sites).

2. **Which site has the highest launch success rate?**
   - **KSC LC-39A** has the highest launch success rate at **76.92%** (10 successes out of 13 attempts).

3. **Which payload range(s) has the highest launch success rate?**
   - The payload range between **2,000 kg and 4,000 kg** (specifically **3,000 kg to 4,000 kg**) demonstrates the highest success rate at **72.7%** (8 successes out of 11 launches).

4. **Which payload range(s) has the lowest launch success rate?**
   - The payload range between **6,000 kg and 8,000 kg** exhibits the lowest success rate at **0%** (0 successes out of 4 launches).

5. **Which F9 Booster version has the highest launch success rate?**
   - **B5 (Block 5)** boasts a **100%** success rate (1 of 1 in this dataset), followed by **FT (Full Thrust)** with a **66.67%** success rate (16 successes out of 24 launches). Early versions `v1.0` (0%) and `v1.1` (6.67%) had the lowest success rates.